# 02 — Exploratory Data Analysis & Text Preprocessing

**Project**: Spam Email Detection  
**Phase**: 2 — EDA & Text Preprocessing  
**Input**: `data/processed/clean_data.csv` (Phase 1 output)  
**Output**: `data/processed/eda_ready.csv` (with clean_message + engineered features)  

| Section | Member attribution |
|---------|-------------------|
| Environment setup & imports | Member 2 (NLP Specialist) |
| Engineered features | Member 4 (Feature Engineer) |
| Text cleaning → clean_message | Member 2 (NLP Specialist) |
| Class distribution plots | Member 1 (Data Lead) |
| Message length plots | Member 1 (Data Lead) |
| WordCloud visualisations | Member 3 (EDA Lead) |
| Top-20 words per class | Member 3 (EDA Lead) |
| Signal features grouped bar chart | Member 4 (Feature Engineer) |
| Save output & summary | Member 1 (Data Lead) |

> ⚠ **Hard rule**: NO `TfidfVectorizer`, NO `CountVectorizer`, NO `train_test_split` in this notebook.  
> Vectorisation happens inside a `Pipeline` in Phase 3, AFTER the split.

---
## Cell 1 — Environment Setup & Imports
**Member 2 (NLP Specialist)**: Bootstrap NLTK resources, import all libraries, load config constants.

In [ ]:
# Member 2 (NLP Specialist): Environment setup
import sys
import os
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# ── Project root on path ──────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import config
from src.nltk_setup import ensure_nltk_resources
from src.preprocessing import clean_text, apply_cleaning

# ── NLTK resource bootstrap ───────────────────────────────────────────────────
print("Checking NLTK resources...")
ensure_nltk_resources(quiet=False)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FIGURES_DIR = os.path.join(PROJECT_ROOT, "reports", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"\nFigures will be saved to: {FIGURES_DIR}")
print(f"Config: RANDOM_STATE={config.RANDOM_STATE}, PROCESSED_DATA_PATH={config.PROCESSED_DATA_PATH}")

---
## Cell 2 — Load Phase 1 Output
**Member 1 (Data Lead)**: Load `clean_data.csv`, verify it meets Phase 1 Definition of Done.

In [ ]:
# Member 1 (Data Lead): Load Phase 1 processed dataset
CLEAN_PATH = os.path.join(PROJECT_ROOT, config.PROCESSED_DATA_PATH)

if not os.path.exists(CLEAN_PATH):
    raise FileNotFoundError(
        f"Phase 1 output not found: {CLEAN_PATH}\n"
        "Run notebooks/01_data_setup_and_audit.ipynb first."
    )

df = pd.read_csv(CLEAN_PATH)

print("=" * 60)
print("LOADED: data/processed/clean_data.csv")
print("=" * 60)
print(f"  Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Columns: {df.columns.tolist()}")
print(f"  Nulls  : {df.isnull().sum().sum()}")
print(f"  Dupes  : {df.duplicated().sum()}")

assert df.isnull().sum().sum() == 0, "Phase 1 output has nulls — re-run notebook 01!"
assert df.duplicated().sum() == 0,   "Phase 1 output has duplicates — re-run notebook 01!"
print("\n  ✅ Phase 1 QC assertions passed.")
df.head()

---
## Cell 3 — Engineered Features
**Member 4 (Feature Engineer)**: Compute deterministic per-row signal features on RAW text.  
These do NOT fit on dataset distribution → no train/test leakage.

In [ ]:
# Member 4 (Feature Engineer): Compute signal features from raw message text

# Regex patterns
_URL_RE       = re.compile(r"https?://|www\.", re.IGNORECASE)
_CURRENCY_RE  = re.compile(r"[\$£₹]")
_DIGIT_RE     = re.compile(r"\d")
_ALLCAPS_RE   = re.compile(r"\b[A-Z]{2,}\b")  # words of 2+ uppercase letters

def _safe_len(s):
    return len(s) if isinstance(s, str) else 0

msg = df[config.TEXT_COL]

# ── Character and word length ─────────────────────────────────────────────────
df["msg_length_chars"] = msg.apply(_safe_len)
df["msg_length_words"] = msg.str.split().str.len().fillna(0).astype(int)

# ── URL / hyperlink presence (0 or 1) ─────────────────────────────────────────
df["has_url"] = msg.apply(
    lambda x: int(bool(_URL_RE.search(str(x))))
)

# ── Currency symbol presence (0 or 1) ────────────────────────────────────────
df["has_currency"] = msg.apply(
    lambda x: int(bool(_CURRENCY_RE.search(str(x))))
)

# ── Digit count and ratio ─────────────────────────────────────────────────────
df["digit_count"] = msg.apply(
    lambda x: len(_DIGIT_RE.findall(str(x)))
)
df["digit_ratio"] = df["digit_count"] / df["msg_length_chars"].clip(lower=1)

# ── ALL-CAPS word count (urgency/shouting — common spam signal) ───────────────
df["uppercase_word_count"] = msg.apply(
    lambda x: len(_ALLCAPS_RE.findall(str(x)))
)

# ── Exclamation mark count ────────────────────────────────────────────────────
df["exclamation_count"] = msg.str.count(r"!")

ENGINEERED_COLS = [
    "msg_length_chars", "msg_length_words",
    "has_url", "has_currency",
    "digit_count", "digit_ratio",
    "uppercase_word_count", "exclamation_count"
]

print("=" * 60)
print("ENGINEERED FEATURES — DESCRIPTIVE STATS BY CLASS")
print("=" * 60)
print(df.groupby(config.LABEL_COL)[ENGINEERED_COLS].mean().round(3).T.to_string())
print(f"\n  ✅ {len(ENGINEERED_COLS)} feature columns added.")

---
## Cell 4 — Text Cleaning → `clean_message`
**Member 2 (NLP Specialist)**: Apply `clean_text()` from `src/preprocessing.py` to produce `clean_message`. Raw `message` is preserved unchanged.

In [ ]:
# Member 2 (NLP Specialist): Apply canonical cleaning pipeline
# remove_numbers=False — digits are spam signals; see docs/preprocessing_notes.md

print("Applying clean_text() pipeline (remove_numbers=False)...")
df["clean_message"] = apply_cleaning(
    df[config.TEXT_COL],
    remove_numbers=False,
    verbose=True
)

print("\n  Before → After examples:")
sample = df[[config.TEXT_COL, "clean_message"]].sample(5, random_state=config.RANDOM_STATE)
for _, row in sample.iterrows():
    print(f"  RAW  : {row[config.TEXT_COL][:80]}")
    print(f"  CLEAN: {row['clean_message'][:80]}")
    print()

---
## Cell 5 — Class Distribution
**Member 1 (Data Lead)**: Bar chart + exact percentages. State imbalance ratio explicitly — it drives Phase 4 metric selection.

In [ ]:
# Member 1 (Data Lead): Class distribution visualisation

counts = df[config.LABEL_COL].value_counts()
pcts   = df[config.LABEL_COL].value_counts(normalize=True) * 100
imbalance_ratio = counts["ham"] / counts["spam"]

print("=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)
for cls in ["ham", "spam"]:
    print(f"  {cls:<6}: {counts[cls]:>5,}  ({pcts[cls]:.1f}%)")
print(f"\n  Imbalance ratio (ham:spam) = {imbalance_ratio:.1f}:1")
print("  → Accuracy alone is misleading; report F1-weighted and spam recall in Phase 4.")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle("Class Distribution — SMS Spam Collection", fontsize=14, fontweight="bold")

# Bar chart
palette = {"ham": "#4C72B0", "spam": "#DD8452"}
ax = axes[0]
bars = ax.bar(counts.index, counts.values,
              color=[palette[c] for c in counts.index], edgecolor="white", width=0.5)
for bar, cnt, pct in zip(bars, counts.values, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{cnt:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=11)
ax.set_xlabel("Class"); ax.set_ylabel("Count")
ax.set_title("Absolute Counts")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=[palette[c] for c in counts.index],
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Proportion")

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "01_class_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print("  Saved: reports/figures/01_class_distribution.png")

---
## Cell 6 — Message Length Distribution
**Member 1 (Data Lead)**: Violin plots for character and word counts, split by class.

In [ ]:
# Member 1 (Data Lead): Message length distributions by class

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Message Length Distribution by Class", fontsize=14, fontweight="bold")

for ax, col, label in zip(
    axes,
    ["msg_length_chars", "msg_length_words"],
    ["Character Count", "Word Count"]
):
    sns.violinplot(
        data=df, x=config.LABEL_COL, y=col,
        palette=palette, inner="quartile", ax=ax, linewidth=1.2
    )
    means = df.groupby(config.LABEL_COL)[col].mean()
    for i, cls in enumerate(["ham", "spam"]):
        ax.axhline(means[cls], color=palette[cls], linestyle="--", linewidth=1,
                   label=f"{cls} mean: {means[cls]:.1f}")
    ax.set_xlabel("Class"); ax.set_ylabel(label)
    ax.set_title(f"{label} by Class")
    ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "02_message_length_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Length stats by class:")
print(df.groupby(config.LABEL_COL)[["msg_length_chars", "msg_length_words"]].describe().round(1).to_string())
print("  Saved: reports/figures/02_message_length_distribution.png")

---
## Cell 7 — Word Clouds
**Member 3 (EDA Lead)**: Separate WordClouds for spam and ham using `clean_message`.

In [ ]:
# Member 3 (EDA Lead): WordCloud visualisations — spam vs ham

def make_wordcloud(text_series, title, colormap, ax):
    corpus = " ".join(text_series.dropna().values)
    wc = WordCloud(
        width=800, height=400,
        background_color="white",
        colormap=colormap,
        max_words=150,
        collocations=False,
        random_state=config.RANDOM_STATE
    ).generate(corpus)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Word Clouds by Class (clean_message)", fontsize=14, fontweight="bold")

spam_text = df.loc[df[config.LABEL_COL] == "spam", "clean_message"]
ham_text  = df.loc[df[config.LABEL_COL] == "ham",  "clean_message"]

make_wordcloud(spam_text, "SPAM messages", "Oranges",  axes[0])
make_wordcloud(ham_text,  "HAM messages",  "Blues",    axes[1])

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "03_wordclouds.png"), dpi=150, bbox_inches="tight")
plt.show()
print("  Saved: reports/figures/03_wordclouds.png")

---
## Cell 8 — Top 20 Most Frequent Words per Class
**Member 3 (EDA Lead)**: Horizontal bar charts using token frequency from `clean_message`.

In [ ]:
# Member 3 (EDA Lead): Top-20 word frequency per class

def get_top_words(text_series, n=20):
    all_words = " ".join(text_series.dropna().values).split()
    return Counter(all_words).most_common(n)

top_spam = get_top_words(spam_text)
top_ham  = get_top_words(ham_text)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Top 20 Most Frequent Words by Class", fontsize=14, fontweight="bold")

for ax, top_words, title, color in zip(
    axes,
    [top_spam, top_ham],
    ["Spam", "Ham"],
    ["#DD8452", "#4C72B0"]
):
    words, freqs = zip(*top_words)
    y_pos = range(len(words))
    ax.barh(y_pos, freqs, color=color, alpha=0.85, edgecolor="white")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(words, fontsize=10)
    ax.invert_yaxis()  # highest at top
    ax.set_xlabel("Frequency")
    ax.set_title(f"Top 20 — {title}", fontsize=12, fontweight="bold")
    for i, (freq, word) in enumerate(zip(freqs, words)):
        ax.text(freq + 0.5, i, str(freq), va="center", fontsize=8)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "04_top20_words.png"), dpi=150, bbox_inches="tight")
plt.show()
print("  Saved: reports/figures/04_top20_words.png")

---
## Cell 9 — Signal Features Grouped Bar Chart
**Member 4 (Feature Engineer)**: Mean `has_url`, `uppercase_word_count`, `exclamation_count` per class.

In [ ]:
# Member 4 (Feature Engineer): Grouped bar chart — mean signal features by class

signal_cols = ["has_url", "has_currency", "uppercase_word_count", "exclamation_count"]
signal_labels = ["Has URL", "Has Currency\nSymbol", "ALL-CAPS\nWord Count", "Exclamation\nCount"]

mean_by_class = df.groupby(config.LABEL_COL)[signal_cols].mean()

x = np.arange(len(signal_cols))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars_ham  = ax.bar(x - width/2, mean_by_class.loc["ham"],  width,
                   label="Ham",  color="#4C72B0", edgecolor="white", alpha=0.9)
bars_spam = ax.bar(x + width/2, mean_by_class.loc["spam"], width,
                   label="Spam", color="#DD8452", edgecolor="white", alpha=0.9)

for bars in [bars_ham, bars_spam]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f"{h:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xlabel("Feature")
ax.set_ylabel("Mean Value per Message")
ax.set_title("Mean Signal Features by Class", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(signal_labels, fontsize=10)
ax.legend(title="Class", fontsize=10)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "05_signal_features_by_class.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Mean signal features by class:")
print(mean_by_class[signal_cols].round(4).to_string())
print("  Saved: reports/figures/05_signal_features_by_class.png")

---
## Cell 10 — Final QC & Save `eda_ready.csv`
**Member 1 (Data Lead)**: Verify output integrity, then save to `data/processed/eda_ready.csv`.

In [ ]:
# Member 1 (Data Lead): Final column selection and QC before save

OUTPUT_COLS = [
    config.LABEL_COL,         # label (string)
    config.LABEL_NUM_COL,     # label_num (int)
    config.TEXT_COL,          # message (raw)
    "clean_message",          # preprocessed text
] + ENGINEERED_COLS

df_out = df[OUTPUT_COLS].copy()

print("=" * 60)
print("FINAL QC — eda_ready.csv")
print("=" * 60)
print(f"  Shape   : {df_out.shape[0]:,} rows × {df_out.shape[1]} columns")
print(f"  Columns : {df_out.columns.tolist()}")

final_nulls = df_out.isnull().sum()
print("\n  Null counts per column:")
for col, n in final_nulls.items():
    flag = "  ⚠ NULLS!" if n > 0 else "  ✓"
    print(f"    {col:<25} {n:>4}{flag}")

assert final_nulls.sum() == 0, "FAIL: nulls in output — investigate!"
print("\n  ✅ Zero nulls. All QC passed.")

In [ ]:
# Member 1 (Data Lead): Save eda_ready.csv

EDA_READY_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "eda_ready.csv")
df_out.to_csv(EDA_READY_PATH, index=False, encoding="utf-8")

# Verify save
df_verify = pd.read_csv(EDA_READY_PATH)
print("=" * 60)
print("SAVE VERIFICATION")
print("=" * 60)
print(f"  Saved to     : {EDA_READY_PATH}")
print(f"  File size    : {os.path.getsize(EDA_READY_PATH) / 1024:.1f} KB")
print(f"  Loaded shape : {df_verify.shape}")
print(f"  Nulls        : {df_verify.isnull().sum().sum()}")
print("\n  ✅ Phase 2 complete. eda_ready.csv is ready for Phase 3.")

print("\n  Figures saved:")
for f in sorted(os.listdir(FIGURES_DIR)):
    if f.endswith(".png"):
        print(f"    reports/figures/{f}")

df_verify.sample(3, random_state=config.RANDOM_STATE)

---
## Phase 2 Summary

| Step | Action | Result |
|------|--------|--------|
| NLTK setup | ensure_nltk_resources() | ✅ |
| Load Phase 1 output | clean_data.csv | ✅ |
| Engineered features | 8 signal columns on raw text | ✅ |
| Text cleaning | clean_message via clean_text() | ✅ |
| Class distribution | Bar + pie, saved PNG | ✅ |
| Message length | Violin plots by class, saved PNG | ✅ |
| WordCloud (spam + ham) | Using clean_message, saved PNG | ✅ |
| Top-20 words | Horizontal bar per class, saved PNG | ✅ |
| Signal features bar chart | Grouped by class, saved PNG | ✅ |
| Save output | data/processed/eda_ready.csv | ✅ |

**No vectorizer fitted. No train_test_split run.**

**Next**: Phase 3 — `03_modelling.ipynb`  
Train-test split → TF-IDF inside Pipeline → MNB + LR training → initial evaluation.